# Install libraries

In [1]:
import pandas as pd
import torch
import os
import numpy as np
import librosa

from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

In [2]:
test_df = pd.read_csv("/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/test.csv")
print("testShape:", test_df.shape)
test_df.head()

testShape: (3020, 2)


,id,filename
0,1,mashups/song0001.wav
1,2,mashups/song0002.wav
2,3,mashups/song0003.wav
3,4,mashups/song0004.wav
4,5,mashups/song0005.wav


In [3]:
from transformers import ASTConfig, ASTForAudioClassification, ASTFeatureExtractor

model_name = "MIT/ast-finetuned-audioset-10-10-0.4593"
feature_extractor = ASTFeatureExtractor.from_pretrained(model_name)

config = ASTConfig.from_pretrained(model_name)
config.num_labels = 10
model = ASTForAudioClassification.from_pretrained(
    model_name, 
    config=config, 
    ignore_mismatched_sizes=True
)

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                          
------------------------+----------+------------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.Size([10])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


In [4]:
import kagglehub
handle = "uditmaurya1588/dlgenai/pytorch/ast-v1"
model_path = kagglehub.model_download(handle)
print("Downloaded:", model_path)

Downloaded: /kaggle/input/models/uditmaurya1588/dlgenai/pytorch/ast-v1/1


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

model_file = "/kaggle/input/models/uditmaurya1588/dlgenai/pytorch/ast-v1/1/Ast_0.93.pth"

state_dict = torch.load(model_file, map_location=device)
model.load_state_dict(state_dict)

print("Model loaded successfully")

Model loaded successfully


# Dataset Class

In [6]:
class Dataset(Dataset):

    def __init__(self, test_csv, mashup_dir, target_w=1024):
        # Read CSV file
        self.df = pd.read_csv(test_csv)

        # Store directory path
        self.mashup_dir = mashup_dir

        # Target width time frames for AST
        self.target_w = target_w

    def __len__(self):
        # Return total number of samples
        return len(self.df)

    def __getitem__(self, idx):
        # Get row from dataframe
        row = self.df.iloc[idx]

        # Get filename safely
        if "filename" in row:
            filename = row["filename"]
        else:
            file_id = str(row["id"])
            filename = file_id.zfill(4) + ".wav"

        # Full file path
        path = os.path.join(self.mashup_dir, filename)

        #Load Audio
        y, sr = librosa.load(path, sr=22050, duration=30.0)

        #Mel Spectrogram
        mel = librosa.feature.melspectrogram(
            y=y,
            sr=22050,
            n_mels=128,
            n_fft=2048,
            hop_length=512
        )

        mel_db = librosa.power_to_db(mel, ref=np.max)

        #  Normalize 
        mean = mel_db.mean()
        std = mel_db.std()

        mel_db = (mel_db - mean) / (std + 1e-6)

        #Transpose
        mel_t = mel_db.T   # shape: (time, freq)

        #1024
        current_length = mel_t.shape[0]

        if current_length > self.target_w:
            start = (current_length - self.target_w) // 2
            end = start + self.target_w
            mel_final = mel_t[start:end, :]
        else:
            padding_needed = self.target_w - current_length
            mel_final = np.pad(mel_t, ((0, padding_needed), (0, 0)))

        # Convert to tensor
        mel_tensor = torch.tensor(mel_final, dtype=torch.float32)

        sample_id = str(row["id"])

        return mel_tensor, sample_id

# Paths and Settings

In [7]:
test_csv = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/test.csv"

mashup_dir = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup"

genres = [
    "blues", "classical", "country", "disco",
    "hiphop", "jazz", "metal", "pop",
    "reggae", "rock"
]

# DataLoader

In [8]:
test_dataset = Dataset(test_csv, mashup_dir)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2
)

# Inference Loop

In [9]:
model.eval()

predictions = []

with torch.no_grad():

    for batch in tqdm(test_loader, desc="Inference"):

        mels = batch[0]
        ids = batch[1]

        # Move to device
        mels = mels.to(device)

        # Model forward pass
        outputs = model(mels)

        # Handle HuggingFace or normal model
        if hasattr(outputs, "logits"):
            logits = outputs.logits
        else:
            logits = outputs

        # Get predicted class index
        preds = torch.argmax(logits, dim=1)

        preds = preds.cpu().numpy()

        # Loop through batch
        for i in range(len(ids)):

            sample_id = ids[i]
            pred_idx = preds[i]

            # Convert id to 4-digit format
            formatted_id = sample_id.zfill(4)

            genre_name = genres[pred_idx]

            result = {
                "id": formatted_id,
                "genre": genre_name
            }

            predictions.append(result)

Inference: 100%|██████████| 189/189 [04:44<00:00,  1.50s/it]


In [10]:
submission = pd.DataFrame(predictions)

submission.to_csv("submission.csv", index=False)

print("Saved", len(submission), "predictions to submission.csv")

Saved 3020 predictions to submission.csv


In [11]:
submission

,id,genre
0,0001,pop
1,0002,jazz
2,0003,disco
3,0004,metal
4,0005,country
...,...,...
3015,3016,rock
3016,3017,jazz
3017,3018,reggae
3018,3019,country
